In [2]:
from functools import partial
import pandas as pd
from datasets import load_dataset


data_path = ["data_raw/train.txt","data_raw/valid.txt","data_raw/test.txt"]
data_output_path = ["data_raw/train.jsonl","data_raw/valid.jsonl","data_raw/test.jsonl"]
from modelscope import GPT2Tokenizer, GPT2LMHeadModel
hf_model_path = 'Fengshenbang/Wenzhong-GPT2-110M-chinese-v2'
tokenizer = GPT2Tokenizer.from_pretrained(hf_model_path)
# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch,tokenizer):
    return tokenizer(batch['context'])

dataset = load_dataset('json', data_files={
    'train': data_output_path[0],
    'valid': data_output_path[1],
    'test': data_output_path[2]
})
# 查看结构
print(dataset['train'][0])
# 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

map_kwargs = {
    'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
    'batch_size': 512,  # 每批 512 个样本
    'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
}

# 在 map 时绑定 tokenizer
tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer)

# 对训练集和验证集应用 map
tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

print(tokenized_dataset_train[0])
print(tokenized_dataset_val[0])



D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.regi

{'idx': 0, 'context': '死囚 爱 刽子手 女贼 爱 衙役 我们 爱 你们 难道 还有 别的 选择 没想到 胡军 除了 蓝宇 还有 东宫 西宫 我 个 去 阿兰 这样 真 他 nia 恶心 爱个 P 分明 只是 欲', 'label': 1}
{'input_ids': [29826, 119, 32368, 248, 13328, 230, 109, 10263, 230, 121, 36310, 33699, 233, 10263, 98, 111, 164, 112, 120, 13328, 230, 109, 5525, 94, 247, 37605, 117, 10545, 230, 239, 20015, 105, 13328, 230, 109, 220, 19526, 254, 20015, 105, 16268, 248, 122, 34402, 241, 5525, 123, 246, 17312, 231, 10263, 230, 104, 21410, 16268, 222, 231, 162, 233, 102, 10545, 110, 94, 46349, 111, 26344, 108, 5525, 225, 94, 37863, 249, 16268, 247, 97, 12859, 228, 5525, 241, 251, 22522, 229, 5525, 123, 246, 17312, 231, 220, 10310, 250, 22522, 104, 5525, 98, 123, 22522, 104, 10545, 230, 239, 220, 10310, 103, 10263, 236, 119, 16268, 246, 123, 17739, 108, 5525, 123, 247, 43718, 115, 13328, 250, 253, 220, 20015, 244, 299, 544, 10545, 223, 114, 33232, 225, 13328, 230, 109, 10310, 103, 350, 10263, 230, 228, 23626, 236, 10263, 237, 103, 42468, 10545, 105, 110], 'attention_mask': [1

In [3]:
for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i+1}: {tokenizer.decode(seq)}')

#清理数据长度国小的
def filter_short_samples(batch, min_length=10):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)
# 对训练集和验证集应用过滤函数
filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

#放入torch
import torch
filtered_dataset_train.set_format(type='torch')
filtered_dataset_val.set_format(type='torch')
print(filtered_dataset_train[0])
print(filtered_dataset_val[0])
# 计算验证集中 input_ids 的最大长度
max_len = max(len(seq) for seq in filtered_dataset_val['input_ids'])
print(f"验证集最大序列长度: {max_len}")


1: 这种 忒 著名 历史 事件 不能 严谨 尊重 历史 一点 除了 预计 撒 自刎 乌江 四面楚歌 细节 没有 太 还原 实属 主要 角色 刘邦 项羽 性格 也 塑造 失败 陈小春 饰演 樊哙 算 性格 鲜明 黎明 上不起 不想 评论 直接 败坏 影片 印象分
2: 导演 你 猪脑子 几场 著名 战役 被 你 拍 动不动 有人 跳出 来说 宋江 哥哥 我 妙计 一条 尼玛 个 妙计 本来 好好 妙计 被 你 给 拍成 屎 不 知道 以为 梁山 一 帮子 爷们儿 脑残 弱智 这样 计 也 敢 称 妙计 怎么着 梁山 也 人才辈出 特技 花
3: 弱智 剧情 没关系 可以 忍 垃圾 剪辑 没关系 可以 忍 东倒西歪 三观 没关系 可以 忍 坑爹 口音 没关系 可以 忍 陈坤 油腻腻 油头 没关系 还是 可以 忍 但是 志玲 姐姐 没有 胸 真的 不能 忍
4: 第一次 看 很小 没见 过 这么 露骨 片子 当 黄片 看 后来 来 英国 之后 又 看 一遍 没什么 意思 萨 朗斯 通 后背 挺 多 肥肉 现在 越开 她 不 顺眼 大妈 到底 谁
5: 我 说 赵氏 孤儿 怎么 那么 拧 巴 想来想去 不 喜欢 陈凯歌 通过 扭曲 青少年 心理 制造 人间 悲剧 叙事 模式 先是 拍 一个 馒头 引发 血案 这次 又 拍 这么 一个 少年 一个 爹 养 他 一个 爹 教 他 一个 爹 他 他 养父 唆使 下 为了 给 他 生父 报仇 杀 他 教父 这是 哪门子 仁义 礼智信
训练集过滤前样本数量: 19998
训练集过滤后样本数量: 3132
验证集过滤前样本数量: 5629
验证集过滤后样本数量: 844
{'input_ids': tensor([29826,   119, 32368,   248, 13328,   230,   109, 10263,   230,   121,
        36310, 33699,   233, 10263,    98,   111,   164,   112,   120, 13328,
          230,   109,  5525,    94,   247, 37605,   117, 10545,   230,   239,
        20015,   105, 13328,   230,   1

In [4]:
#填充
print(tokenizer.pad_token)
print(tokenizer.eos_token)
#将填充标记设置为 EOS 标记
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling
# mlm=False，将数据整理成“因果语言建模”需要的数据格式
# “因果语言建模”就是“预测下一个token”类型的任务，也就是gpt风格的自回归模型
# 如果mlm=True，那么数据整理成bert风格的任务所需的数据格式
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False) # labels

dataloader_params = {
    'batch_size': 16, #根据需要调整批量大小
    'collate_fn': data_collator
}

train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

print(len(train_dataloader))

batch = next(iter(train_dataloader))
print(batch.keys())
print(batch['input_ids'].shape)
print(batch['input_ids'][0])
print(batch['labels'][0])
print(batch['attention_mask'][0])

None
<|endoftext|>
196
dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([16, 150])
tensor([29826,   119, 32368,   248, 13328,   230,   109, 10263,   230,   121,
        36310, 33699,   233, 10263,    98,   111,   164,   112,   120, 13328,
          230,   109,  5525,    94,   247, 37605,   117, 10545,   230,   239,
        20015,   105, 13328,   230,   109,   220, 19526,   254, 20015,   105,
        16268,   248,   122, 34402,   241,  5525,   123,   246, 17312,   231,
        10263,   230,   104, 21410, 16268,   222,   231,   162,   233,   102,
        10545,   110,    94, 46349,   111, 26344,   108,  5525,   225,    94,
        37863,   249, 16268,   247,    97, 12859,   228,  5525,   241,   251,
        22522,   229,  5525,   123,   246, 17312,   231,   220, 10310,   250,
        22522,   104,  5525,    98,   123, 22522,   104, 10545,   230,   239,
          220, 10310,   103, 10263,   236,   119, 16268,   246,   123, 17739,
          108,  5525,   123,   247, 43718,  

In [5]:
#加载模型监督微调
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT2LMHeadModel.from_pretrained(hf_model_path).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
#训练一个epoch
num_epochs = 1

In [9]:
# 训练循环
#先评估一下初始模型在验证集上的表现
def validate(epoch):
    model.eval()
    total_loss = 0.0
    for i, batch in enumerate(val_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = model(**batch)
            loss = outputs.loss # 损失
            total_loss += loss.item()
    print(f'val_loss at {epoch} epoch:', total_loss / len(val_dataloader))
validate(1)
#正式训练
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss # 损失
        total_loss += loss.item()
        optimizer.zero_grad() # 梯度清零
        loss.backward() # 反向传播
        optimizer.step() # 更新参数
        print(f'train_loss at {i} epoch:', loss.item() / 16)
    print("train_loss at {epoch} epoch:", total_loss / len(train_dataloader))
    validate(epoch+1)

val_loss at 1 epoch: 1.7431264238537483
train_loss at 0 epoch: 0.07437413185834885
train_loss at 1 epoch: 0.0741664320230484
train_loss at 2 epoch: 0.07549436390399933
train_loss at 3 epoch: 0.07162542641162872
train_loss at 4 epoch: 0.07542207092046738
train_loss at 5 epoch: 0.07091182470321655
train_loss at 6 epoch: 0.06675805896520615
train_loss at 7 epoch: 0.06631536036729813
train_loss at 8 epoch: 0.07565917074680328
train_loss at 9 epoch: 0.0707969069480896
train_loss at 10 epoch: 0.07127133756875992
train_loss at 11 epoch: 0.07169972360134125
train_loss at 12 epoch: 0.07731105387210846
train_loss at 13 epoch: 0.07057958841323853
train_loss at 14 epoch: 0.0681609958410263
train_loss at 15 epoch: 0.07181770354509354
train_loss at 16 epoch: 0.08247096091508865
train_loss at 17 epoch: 0.06868879497051239
train_loss at 18 epoch: 0.07675210386514664
train_loss at 19 epoch: 0.07469309121370316
train_loss at 20 epoch: 0.07380326837301254
train_loss at 21 epoch: 0.0726262554526329
train_

In [10]:
#保存微调后的模型
model.save_pretrained('models/fine_tuned_gpt2')
tokenizer.save_pretrained('models/fine_tuned_gpt2')

('models/fine_tuned_gpt2\\tokenizer_config.json',
 'models/fine_tuned_gpt2\\special_tokens_map.json',
 'models/fine_tuned_gpt2\\vocab.json',
 'models/fine_tuned_gpt2\\merges.txt',
 'models/fine_tuned_gpt2\\added_tokens.json')